# 05 — The −95 minute q99 precursor motif

## Why this module exists

After extending q99 event anatomy to ±120 minutes, a visually coherent pattern appeared around roughly **95 minutes before q99 onset** in several feature families.

This module does not assume that one exact candle is magical. It freezes a five-candle local window:

\[
-105,-100,-95,-90,-85\text{ minutes}
\]

and asks whether the shape is repeatable.

Because this motif was discovered by looking at the same sample, all results here are **exploratory characterization**, not confirmatory inference or out-of-sample prediction.

## Main protection: prior-q99 contamination

A q99 event can occur less than two hours after another q99 event. Such cases could make an apparent −95m pattern nothing more than the aftermath of the earlier shock.

The primary Module 05 sample therefore requires:

\[
\text{no prior q99 bar in }[t_0-120m,t_0).
\]

The all-event sample is retained only as a comparison.


In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_q99_precursors as qp
from data_engineering.modeling import volatility_q99_conditional as qc
from data_engineering.modeling import volatility_q99_motif95 as qm

START = os.getenv('VD5_START', '').strip() or None
END = os.getenv('VD5_END', '').strip() or None
ASSET_FILTER = [x.strip().upper() for x in os.getenv('VD5_ASSETS', '').split(',') if x.strip()]
MAX_CONTROLS = int(os.getenv('VD5_MAX_CONTROLS', str(qc.DEFAULT_MAX_CONTROLS)))
MIN_CONTROLS = int(os.getenv('VD5_MIN_CONTROLS', str(qc.DEFAULT_MIN_CONTROLS)))
MIN_ENV_DIMS = int(os.getenv('VD5_MIN_ENV_DIMS', str(qc.DEFAULT_MIN_ENV_DIMS)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '05'
OUT.mkdir(parents=True, exist_ok=True)

print('Module 03 engine:', qp.VERSION)
print('Module 04 engine:', qc.VERSION)
print('Module 05 engine:', qm.VERSION)
print('Module 05 selftest:', qm.selftest())


## 1. Frozen motif feature contract

The motif study does not reopen the 205-feature search. It carries forward the Module 03/04 candidate families that motivated the visual observation:

- native range / trading activity;
- option IV levels;
- spread / book activity;
- L2 zero-cross / transition / roughness instability.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
schema = con.execute('DESCRIBE time_series_feature_state').df()
available = set(schema['column_name'].astype(str))
contract = qm.motif_contract(available)
display(contract)
features = contract.loc[contract.present, 'feature'].tolist()
missing_features = contract.loc[~contract.present, 'feature'].tolist()
if missing_features:
    print('missing motif features:', missing_features)


## 2. Load raw q99 source plus frozen motif fields


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

select_cols = list(dict.fromkeys(['state_time','underlying','return_5m'] + features))
where = []
if START:
    ts = pd.Timestamp(START)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
if END:
    ts = pd.Timestamp(END)
    ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
    end_ts = ts + pd.Timedelta(days=1)
    where.append(f"state_time < TIMESTAMPTZ '{end_ts.isoformat()}'")
if ASSET_FILTER:
    where.append('underlying IN (' + ','.join(repr(x) for x in ASSET_FILTER) + ')')
where_sql = '' if not where else ' WHERE ' + ' AND '.join(where)
sql = (
    'SELECT ' + ', '.join(qident(c) for c in select_cols)
    + ' FROM time_series_feature_state' + where_sql
    + ' ORDER BY underlying, state_time'
)
df = con.execute(sql).df()
con.close()
df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
df = df.reset_index(drop=True)
if df.empty:
    raise RuntimeError('No rows found.')
dupes = int(df.duplicated(['underlying','state_time']).sum())
if dupes:
    raise RuntimeError(f'Found {dupes} duplicate asset/timestamp rows.')
print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())}')
print(df.state_time.min(), '->', df.state_time.max())


## 3. Recompute q99 and audit prior-q99 contamination


In [ ]:
events, event_meta = qp.identify_q99_onsets(df)
origins = qp.build_origin_index(df, events)
clean_events, clean_origins, contamination = qm.clean_event_and_origin_sets(
    df, events, origins, lookback_min=qm.PRIOR_Q99_LOOKBACK_MIN
)
if clean_events.empty:
    raise RuntimeError('No clean q99 events remain after prior-q99 exclusion.')

summary = pd.DataFrame({
    'sample': ['all_q99_onsets','clean_no_prior_q99_120m'],
    'n_asset_events': [len(events), len(clean_events)],
    'n_unique_event_times': [events.event_time.nunique(), clean_events.event_time.nunique()],
})
display(summary)
display(contamination.prior_q99_bar_lookback.value_counts(dropna=False).rename_axis('prior_q99').to_frame('n_events'))
display(contamination[contamination.prior_q99_bar_lookback].minutes_since_latest_prior_q99.describe())


## 4. Rebuild Module 04 native-volatility matched controls

The all-event sample and the clean sample are both matched independently. Clean controls are also required to have no prior q99 bar in their own preceding 120 minutes.


In [ ]:
all_controls, all_support = qc.choose_volatility_matched_controls(
    df, origins, events,
    max_controls=MAX_CONTROLS,
    min_env_dims=MIN_ENV_DIMS,
)
clean_controls, clean_support = qc.choose_volatility_matched_controls(
    df, clean_origins, clean_events,
    max_controls=MAX_CONTROLS,
    min_env_dims=MIN_ENV_DIMS,
)
print('all events with >= min controls:', int((all_support.n_controls_selected >= MIN_CONTROLS).sum()), '/', len(all_support))
print('clean events with >= min controls:', int((clean_support.n_controls_selected >= MIN_CONTROLS).sum()), '/', len(clean_support))
display(clean_support[['n_eligible_same_clock','n_vol_comparable','n_controls_selected','median_selected_env_distance']].describe())


## 5. Reconstruct the full pre-event profile to t0

Percentiles are measured against each origin's native-volatility-matched controls. The plotted range is −120m through the q99 event candle, but Module 05 motif statistics use only −105m through −85m.


In [ ]:
all_scores = qp.percentile_scores(
    df, features, events, all_controls, qm.PROFILE_BARS,
    min_controls=MIN_CONTROLS,
)
clean_scores = qp.percentile_scores(
    df, features, clean_events, clean_controls, qm.PROFILE_BARS,
    min_controls=MIN_CONTROLS,
)
if clean_scores.empty:
    raise RuntimeError('No usable clean motif comparisons.')
all_profiles = qp.summarize_profiles(all_scores)
clean_profiles = qp.summarize_profiles(clean_scores)
print('all comparisons:', f'{len(all_scores):,}')
print('clean comparisons:', f'{len(clean_scores):,}')


## 6. The five-candle −95m local motif

For each feature define:

\[
Pre=mean(P_{-105},P_{-100}),
\]
\[
Center=P_{-95},
\]
\[
Post=mean(P_{-90},P_{-85}).
\]

Then measure:

- window level = mean of all five percentiles;
- center-vs-neighbors = (Center-(Pre+Post)/2);
- rebound = (Post-Center).

This distinguishes a persistent elevated level from a local notch/reacceleration shape.


In [ ]:
local_all = qm.local_motif_event_table(all_scores)
local_clean = qm.local_motif_event_table(clean_scores)
summary_all = qm.summarize_local_motif(local_all).add_prefix('all_').rename(columns={'all_feature':'feature'})
summary_clean = qm.summarize_local_motif(local_clean).add_prefix('clean_').rename(columns={'clean_feature':'feature'})
motif_compare = summary_all.merge(summary_clean, on='feature', how='outer')
display(motif_compare.sort_values('clean_n_unique_event_times', ascending=False).round(4))


## 7. Plot the suspected shape around −95m

Solid lines use the clean sample. The dashed vertical line marks −95 minutes; the shaded conceptual window is −105 to −85 minutes.


In [ ]:
for feature in features:
    g = clean_profiles[clean_profiles.feature == feature].sort_values('rel_min')
    if not len(g):
        continue
    fig, ax = plt.subplots(figsize=(9,3))
    ax.plot(g.rel_min, g.mean_percentile, marker='o', markersize=2)
    ax.axhline(0.5, linewidth=1)
    ax.axvline(-95, linewidth=1, linestyle='--')
    ax.axvspan(-105, -85, alpha=0.10)
    ax.set_xlim(-120, 0)
    ax.set_ylim(0,1)
    ax.set_title(feature + ' — clean q99 onsets')
    ax.set_xlabel('minutes before q99 onset')
    ax.set_ylabel('mean percentile vs vol-matched controls')
    plt.show()


## 8. Family-level motif components

To avoid making a claim from one feature, construct event-level family summaries.

`activity_level`, `iv_level`, and `liquidity_book_activity_level` are average matched-control percentiles through the five-candle motif window.

`l2_rebound_95_to_post` asks whether L2 instability increases from the −95 candle into the −90/−85 candles.


In [ ]:
components = qm.motif_component_scores(clean_scores, contract)
component_summary = qm.summarize_components(components)
bootstrap = qm.bootstrap_component_means(components)
loo = qm.leave_one_asset_out(components)
display(component_summary.round(4))
display(bootstrap.round(4))


## 9. Leave-one-asset-out stability

A useful cross-asset motif should not disappear when one instrument is removed. Sparse L2 components should be interpreted especially cautiously because their event support is much smaller.


In [ ]:
for component in sorted(loo.component.unique()) if len(loo) else []:
    g = loo[loo.component == component]
    print('\n', component)
    display(g[['excluded_asset','n_unique_event_times','mean','mean_minus_neutral']].round(4))


## 10. Asset-level view of the exact −95 candle


In [ ]:
center_asset = (
    clean_scores[clean_scores.rel_bar == qm.MOTIF_CENTER_BAR]
    .groupby(['underlying','feature'])
    .agg(
        n_events=('event_id','nunique'),
        mean_center_95m_percentile=('percentile_vs_controls','mean'),
        median_center_95m_percentile=('percentile_vs_controls','median'),
        median_robust_diff_iqr=('robust_diff_iqr','median'),
    )
    .reset_index()
)
display(center_asset.sort_values(['feature','underlying']).round(4))


## 11. Persist compact outputs


In [ ]:
contract.to_csv(OUT / '05_motif_contract.csv', index=False)
event_meta.to_csv(OUT / '05_q99_thresholds_and_counts.csv', index=False)
summary.to_csv(OUT / '05_event_sample_summary.csv', index=False)
contamination.to_csv(OUT / '05_prior_q99_contamination.csv', index=False)
clean_events.to_csv(OUT / '05_clean_event_origins.csv', index=False)
clean_support.to_csv(OUT / '05_clean_match_support.csv', index=False)
clean_controls.to_csv(OUT / '05_clean_vol_matched_controls.csv', index=False)
all_profiles.to_csv(OUT / '05_all_event_profiles.csv', index=False)
clean_profiles.to_csv(OUT / '05_clean_event_profiles.csv', index=False)
local_all.to_csv(OUT / '05_all_local_motif_event_table.csv', index=False)
local_clean.to_csv(OUT / '05_clean_local_motif_event_table.csv', index=False)
motif_compare.to_csv(OUT / '05_motif_all_vs_clean.csv', index=False)
components.to_csv(OUT / '05_clean_motif_event_components.csv', index=False)
component_summary.to_csv(OUT / '05_component_summary.csv', index=False)
bootstrap.to_csv(OUT / '05_component_bootstrap.csv', index=False)
loo.to_csv(OUT / '05_leave_one_asset_out.csv', index=False)
center_asset.to_csv(OUT / '05_center95_by_asset.csv', index=False)

meta = {
    'version': qm.VERSION,
    'module': '05_minus95m_q99_motif',
    'motivation': 'visual pattern around -95m after extending q99 anatomy to +/-120m',
    'post_selection_exploratory': True,
    'q99_recomputed_inside_module': True,
    'motif_window_minutes': [-105,-100,-95,-90,-85],
    'motif_center_minutes': -95,
    'primary_sample': 'q99 onsets with no prior q99 bar in previous 120 minutes',
    'control_matching': 'same asset + exact 5m UTC clock + native-volatility history matching from Module 04',
    'simultaneous_asset_weighting': 'collapsed by exact event timestamp for pooled summaries',
    'bootstrap_note': 'descriptive stability only; motif selected after viewing same data',
    'no_predictive_model': True,
    'no_trading_claim': True,
    'start': START or 'earliest_available',
    'end': END or 'latest_available',
    'n_rows': int(len(df)),
    'n_all_asset_events': int(len(events)),
    'n_clean_asset_events': int(len(clean_events)),
    'n_all_unique_event_times': int(events.event_time.nunique()),
    'n_clean_unique_event_times': int(clean_events.event_time.nunique()),
}
(OUT / '05_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote Module 05 outputs to', OUT)


## Interpretation boundary

Module 05 can tell us whether the visually discovered −95m motif is stable, cross-asset, and still present after removing events preceded by another q99 shock.

It **cannot** establish that the motif predicts q99 prospectively, because the −95m window was discovered on this same historical sample. If the clean motif survives, the next legitimate step is to freeze its exact definition and test it chronologically on future/non-discovery observations.
